In [2]:
# %pip install xgboost numpy pandas matplotlib scikit-learn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor


In [17]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

def custom_mse(y_true,y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    n = y_true.shape[0]
    residuals = y_true - y_pred          
    squared_residuals = residuals ** 2  
    sum_squared = np.sum(squared_residuals)  # step 3
    mse = sum_squared / n                # step 4
    return float(mse)
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)


SEED=445
df=pd.read_csv('./datasets/RG-Wage.csv')
df=df.dropna()
df = df.drop(columns=["logwage", "region"])
print(df.shape)

X=df.drop(columns=["wage"])
y=df["wage"]
X_train, X_temp, y_train, y_temp = train_test_split(X,y,test_size=0.30,random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(X_temp,y_temp,test_size=0.50,random_state=SEED)

X_train_encoded = encoder.fit_transform(X_train)
X_val_encoded = encoder.transform(X_val)
X_test_encoded = encoder.transform(X_test)

# Descion tree
DT_Hyperparameters = {
    "max_depth": [2, 3, 4, 5, 6, 7, 8, 9, 10],
    "min_samples_split": [2, 5, 10, 20],    
    "min_samples_leaf": [1, 2, 4, 8],
    "criterion": ["squared_error", "absolute_error"]
}
results=[]
best_mse=1e10 #theoriticially an infinite number for comaparisoon
best_performing_model=None
best_hyperparameters=None
for depth in DT_Hyperparameters["max_depth"]:
    for min_samples_split in DT_Hyperparameters["min_samples_split"]:
        for min_samples_leaf in DT_Hyperparameters["min_samples_leaf"]:
            for criterion in DT_Hyperparameters["criterion"]:
                model = DecisionTreeRegressor(max_depth=depth, 
                                              min_samples_split=min_samples_split, 
                                              min_samples_leaf=min_samples_leaf, 
                                              criterion=criterion, random_state=SEED)
                model.fit(X_train_encoded, y_train)
                y_pred = model.predict(X_val_encoded)
                mse = custom_mse(y_val, y_pred)
                results.append((depth, min_samples_split, min_samples_leaf, criterion, mse))
                if mse < best_mse:
                    best_mse = mse
                    best_performing_model = model
                    best_hyperparameters = (depth, min_samples_split, min_samples_leaf, criterion)

results_df=(pd.DataFrame(results, columns=["max_depth", "min_samples_split", "min_samples_leaf", "criterion", "mse"])
 .sort_values(by="mse")
 .reset_index(drop=True)
)
print("Best Hyperparameters:", best_hyperparameters)
print("Best MSE:", best_mse)
print("Model:", best_performing_model)

(3000, 9)
Best Hyperparameters: (4, 2, 8, 'squared_error')
Best MSE: 1271.0710775562552
Model: DecisionTreeRegressor(max_depth=4, min_samples_leaf=8, random_state=445)


In [19]:
XGB_GRID = {
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.05, 0.1],
    "n_estimators": [100, 300],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "min_child_weight": [1, 5],
    "gamma": [0, 0.1],
}
results = []

best_mse = float("inf")
best_model = None
best_params = None

rng = np.random.RandomState(SEED)

for _ in range(60):
    max_depth = rng.choice(XGB_GRID["max_depth"])
    learning_rate = rng.choice(XGB_GRID["learning_rate"])
    n_estimators = rng.choice(XGB_GRID["n_estimators"])
    subsample = rng.choice(XGB_GRID["subsample"])
    colsample_bytree = rng.choice(XGB_GRID["colsample_bytree"])

    model = XGBRegressor(
        objective="reg:squarederror",
        random_state=SEED,
        n_jobs=4,
        verbosity=0,
        max_depth=max_depth,
        learning_rate=learning_rate,
        n_estimators=n_estimators,
        subsample=subsample,
        colsample_bytree=colsample_bytree
    )

    model.fit(X_train_encoded, y_train)
    predictions = model.predict(X_val_encoded)
    mse = custom_mse(y_val, predictions)

    results.append({
        "max_depth": max_depth,
        "learning_rate": learning_rate,
        "n_estimators": n_estimators,
        "subsample": subsample,
        "colsample_bytree": colsample_bytree,
        "validation_mse": mse
    })
    if mse < best_mse:
        best_mse = mse
        model2 = model
        best_params = {
            "max_depth": max_depth,
            "learning_rate": learning_rate,
            "n_estimators": n_estimators,
            "subsample": subsample,
            "colsample_bytree": colsample_bytree
        }

results_df = pd.DataFrame(results).sort_values("validation_mse").reset_index(drop=True)

print("Best Hyperparameters:", best_params)
print("Best MSE:", best_mse)
print("Model:", model2)

Best Hyperparameters: {'max_depth': np.int64(3), 'learning_rate': np.float64(0.05), 'n_estimators': np.int64(100), 'subsample': np.float64(1.0), 'colsample_bytree': np.float64(0.8)}
Best MSE: 1244.292571772576
Model: XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=np.float64(0.8), device=None,
             early_stopping_rounds=None, enable_categorical=True,
             eval_metric=None, feature_types=None, feature_weights=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=np.float64(0.05),
             max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=np.int64(3), max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=np.int64(100), n_jobs=4,
             num_parallel_t

In [ ]:
# Confusion matrix 
def custom_confusion_matrix(y_actual, y_predicted):
    # converting to numpy array for efficiency
    y_true = np.array(y_actual,dtype=int)
    y_pred = np.array(y_predicted,dtype=int)

    trueP = int(np.sum((y_pred == 1) & (y_true == 1)))
    trueN = int(np.sum((y_pred == 0) & (y_true == 0)))
    falseP = int(np.sum((y_pred == 1) & (y_true == 0)))
    falseN = int(np.sum((y_pred == 0) & (y_true == 1)))
    return trueP, trueN, falseP, falseN

def custom_accuracy(y_actual, y_predicted):
    tp, tn, fp, fn = custom_confusion_matrix(y_actual, y_predicted)  
    if tp + tn + fp + fn == 0:
            return 0
    else:
        correct = tp + tn                                        
        total = tp + tn + fp + fn
        return correct / total              

def custom_precision(y_actual, y_predicted):
    tp, tn, fp, fn = custom_confusion_matrix(y_actual, y_predicted)
    denom = tp + fp
    if denom == 0:
        return 0.0 
    else:
        return tp / denom 


def custom_recall(y_actual, y_predicted):
    
    tp, tn, fp, fn = custom_confusion_matrix(y_actual, y_predicted)
    denom = tp + fn
    return tp / denom if denom > 0 else 0.0


def custom_f1(y_actual, y_predicted):
    """F1 = 2 * Precision * Recall / (Precision + Recall), with 0.0 if undefined."""
    p = custom_precision(y_actual, y_predicted)
    r = custom_recall(y_actual, y_predicted)
    return (2 * p * r / (p + r)) if (p + r) > 0 else 0.0


def custom_classification_report(y_actual, y_predicted):
    tp, tn, fp, fn = custom_confusion_matrix(y_actual, y_predicted)
    return {
        "accuracy": custom_accuracy(y_actual, y_predicted),
        "precision": custom_precision(y_actual, y_predicted),
        "recall": custom_recall(y_actual, y_predicted),
        "f1": custom_f1(y_actual, y_predicted),
        "confusion_matrix": {"TP": tp, "TN": tn, "FP": fp, "FN": fn},
    }
